# Classical Baseline — Quantum vs Classical Comparison

**InclusiFund Research Vault**
**Status:** Experimental
**Data:** SYNTHETIC ONLY — no client data

---

## Objective

Train classical ML models (Logistic Regression, Random Forest, SVM, Gradient Boosting)
and compare against the quantum grant matcher to measure if quantum offers any advantage
on this task.

In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve
)

from quantum_grants.data.synthetic import generate_dataset
from quantum_grants.models.feature_encoder import encode_batch
from quantum_grants.training.trainer import train, TrainingConfig

print('Modules loaded successfully')

Modules loaded successfully


## 1. Generate Dataset & Encode Features

In [2]:
grants, applicants, labels = generate_dataset(n_grants=20, n_applicants=40, seed=42)

# Build feature matrix and label vector
X = encode_batch(grants, applicants)
y = np.array([l.is_match for l in labels], dtype=int)

print(f'Feature matrix: {X.shape}')
print(f'Labels: {y.shape} — {y.sum()} positive ({100*y.mean():.1f}%)')

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape[0]} samples, Test: {X_test.shape[0]} samples')

Generated 20 grants x 40 applicants = 800 pairs
Positive matches: 80 (10.0%)
Feature matrix: (800, 6)
Labels: (800,) — 80 positive (10.0%)
Train: 640 samples, Test: 160 samples


## 2. Train Classical Models

In [3]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM (RBF)': SVC(kernel='rbf', probability=True, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
}

classical_results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    classical_results[name] = {
        'model': model,
        'y_pred': y_pred,
        'y_proba': y_proba,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'f1': f1_score(y_test, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_test, y_proba),
    }
    print(f'{name}: acc={classical_results[name]["accuracy"]:.4f}, f1={classical_results[name]["f1"]:.4f}, auc={classical_results[name]["roc_auc"]:.4f}')

Logistic Regression: acc=0.9875, f1=0.9412, auc=0.9957
Random Forest: acc=0.9938, f1=0.9677, auc=1.0000
SVM (RBF): acc=0.9875, f1=0.9412, auc=0.9991
Gradient Boosting: acc=1.0000, f1=1.0000, auc=1.0000


## 3. Train Quantum Model

In [4]:
config = TrainingConfig(
    n_layers=2,
    learning_rate=0.05,
    n_epochs=15,
    batch_size=16,
    n_grants=20,
    n_applicants=40,
    seed=42,
)

quantum_logs = train(config)
final = quantum_logs[-1]

print(f'Quantum model: acc={final.accuracy:.4f}, loss={final.loss:.4f}')
print(f'(Note: quantum accuracy is on training set — not directly comparable to test set metrics)')

QUANTUM GRANT MATCHER — TRAINING (R&D)
Layers: 2
LR: 0.05
Epochs: 15
Dataset: 20 grants x 40 applicants

Generated 20 grants x 40 applicants = 800 pairs
Positive matches: 80 (10.0%)


/Users/royalreece/Desktop/Claude Code: AntiGravity Projects/InclusiFund-Research/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Epoch   1/15 | Loss: 0.7593 | Acc: 62.50% | Time: 2.2s
Epoch   2/15 | Loss: 0.7970 | Acc: 87.50% | Time: 1.9s
Epoch   3/15 | Loss: 0.8483 | Acc: 68.75% | Time: 3.2s
Epoch   4/15 | Loss: 0.8298 | Acc: 62.50% | Time: 2.9s
Epoch   5/15 | Loss: 0.7615 | Acc: 87.50% | Time: 3.0s
Epoch   6/15 | Loss: 0.7812 | Acc: 75.00% | Time: 2.3s
Epoch   7/15 | Loss: 0.7082 | Acc: 87.50% | Time: 3.7s
Epoch   8/15 | Loss: 0.8121 | Acc: 87.50% | Time: 2.6s
Epoch   9/15 | Loss: 0.8190 | Acc: 56.25% | Time: 2.1s
Epoch  10/15 | Loss: 0.7223 | Acc: 87.50% | Time: 2.6s
Epoch  11/15 | Loss: 0.7689 | Acc: 75.00% | Time: 2.2s
Epoch  12/15 | Loss: 0.7648 | Acc: 75.00% | Time: 2.9s
Epoch  13/15 | Loss: 0.7717 | Acc: 87.50% | Time: 2.8s
Epoch  14/15 | Loss: 0.7376 | Acc: 68.75% | Time: 2.1s
Epoch  15/15 | Loss: 0.7655 | Acc: 87.50% | Time: 2.5s

Training log saved to: Research/Training Logs/training_20260311_223131.json
Quantum model: acc=0.8750, loss=0.7655
(Note: quantum accuracy is on training set — not directly c

## 4. Comparison Table

In [5]:
rows = []
for name, r in classical_results.items():
    rows.append({
        'Model': name,
        'Type': 'Classical',
        'Accuracy': r['accuracy'],
        'Precision': r['precision'],
        'Recall': r['recall'],
        'F1': r['f1'],
        'ROC-AUC': r['roc_auc'],
    })

rows.append({
    'Model': 'Quantum (2-layer VQC)',
    'Type': 'Quantum',
    'Accuracy': final.accuracy,
    'Precision': None,
    'Recall': None,
    'F1': None,
    'ROC-AUC': None,
})

comparison_df = pd.DataFrame(rows).set_index('Model')
print(comparison_df.to_string())

                            Type  Accuracy  Precision  Recall        F1   ROC-AUC
Model                                                                            
Logistic Regression    Classical   0.98750   0.888889  1.0000  0.941176  0.995660
Random Forest          Classical   0.99375   1.000000  0.9375  0.967742  1.000000
SVM (RBF)              Classical   0.98750   0.888889  1.0000  0.941176  0.999132
Gradient Boosting      Classical   1.00000   1.000000  1.0000  1.000000  1.000000
Quantum (2-layer VQC)    Quantum   0.87500        NaN     NaN       NaN       NaN


## 5. ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']

for (name, r), color in zip(classical_results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, r['y_proba'])
    ax.plot(fpr, tpr, color=color, lw=2,
            label=f'{name} (AUC={r["roc_auc"]:.3f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — Classical Models')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. F1 Score Comparison

In [ ]:
model_names = list(classical_results.keys())
f1_scores = [classical_results[n]['f1'] for n in model_names]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(model_names, f1_scores, color=colors, edgecolor='black', alpha=0.8)

# Add quantum reference line
ax.axhline(y=final.accuracy, color='red', linestyle='--', lw=2,
           label=f'Quantum accuracy={final.accuracy:.3f} (train set)')

for bar, score in zip(bars, f1_scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{score:.3f}', ha='center', fontsize=11, fontweight='bold')

ax.set_ylabel('F1 Score')
ax.set_title('F1 Score Comparison — Classical Models vs Quantum Reference')
ax.set_ylim(0, 1.1)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## Conclusion

### Key Findings

1. **Classical baselines** provide a strong reference point for the grant matching task
2. **Random Forest and Gradient Boosting** typically perform best on tabular data with 6 features
3. **Quantum model** accuracy is measured on training data (not test), so direct comparison requires
   implementing a quantum test-set evaluation pipeline

### Quantum Advantage Assessment

At this scale (6 features, ~800 pairs), classical models are expected to match or outperform
quantum approaches. Potential quantum advantage may emerge with:
- Higher-dimensional feature spaces
- Non-linear entanglement patterns in real grant data
- Larger-scale optimisation problems

### Next Steps

1. Implement quantum test-set evaluation for fair comparison
2. Test on real Convex grant data (93 grants now available)
3. Increase feature dimensionality to explore quantum expressivity
4. Run on Origin Quantum Cloud hardware for noise-aware benchmarking

---
*See notebook 02 for hyperparameter sweep results.*